# Notebook 01 - Coletor 1001tracklists

Use para:
1. **Testar** o scraping de um set individual
2. **Testar** a lista de URLs de um genero
3. **Coletar** lotes pequenos para validacao
4. **Monitorar** o status da base

> Para coleta em escala (1000+ sets), use o GitHub Actions.

In [ ]:
import sys
sys.path.insert(0, '../')

import os, logging
from dotenv import load_dotenv
from supabase import create_client
from src.collector.collector import Collector

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
load_dotenv('../.env')

sb        = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_KEY'])
collector = Collector(sb)
print('Coletor pronto')
# NOTA: o collector usa async_playwright.
# Em Jupyter, use 'await' diretamente nas celulas abaixo.
# Exemplo: urls = await collector.get_set_urls('goa-psy-trance', 20)

## TESTE 1: Scraping de um set individual

Valida que o Playwright consegue acessar o site. Leva ~15 segundos.

In [ ]:
TEST_URL = 'https://www.1001tracklists.com/tracklist/2tlglsf9/ben-bohmer-abgt-300-asiaworld-expo-hong-kong-2018-09-29.html'

print('Testando scraping... aguarde ~15s')
set_data = await collector.scrape_set(TEST_URL)

if set_data:
    print(f'OK - DJ: {set_data["dj_name"]}')
    print(f'   Titulo: {set_data["set_title"]}')
    print(f'   Data: {set_data["set_date"]}')
    print(f'   Tracks encontradas: {len(set_data["tracks"])}')
    print()
    print('Primeiras 5 tracks:')
    for t in set_data['tracks'][:5]:
        print(f"  {t['position']:02d}. {t['artist']} - {t['title']}")
else:
    print('Retornou vazio. Possiveis causas:')
    print('  1. Site bloqueou o acesso (tente novamente em alguns minutos)')
    print('  2. Estrutura HTML mudou (envie o erro para correcao)')

## TESTE 2: Lista de URLs de um genero

Valida que conseguimos navegar na pagina de genero.

In [ ]:
GENERO = 'goa-psy-trance'

print(f'Buscando URLs de {GENERO}... aguarde ~20s')
urls = await collector.get_set_urls(GENERO, max_sets=20)

print(f'{len(urls)} URLs encontradas:')
for u in urls[:10]:
    print(f'  {u}')

## Coleta pequena (pipeline completo com Supabase)

Coleta 10 sets e salva no Supabase. Use para confirmar que o pipeline inteiro funciona.

In [ ]:
GENERO   = 'goa-psy-trance'
MAX_SETS = 10

print(f'Coletando {MAX_SETS} sets de {GENERO}...')
stats = await collector.collect_genre(GENERO, max_sets=MAX_SETS)

print('Resultado:')
print(f'  Coletados:   {stats["collected"]}')
print(f'  Ja existiam: {stats["skipped"]}')
print(f'  Erros:       {stats["errors"]}')

## Status atual da base

In [ ]:
import pandas as pd

generos = sb.table('genres').select('id, name').eq('active', True).execute().data
rows = []
for g in generos:
    s = sb.table('sets').select('id', count='exact').eq('genre_id', g['id']).execute().count or 0
    t = sb.table('transitions').select('id', count='exact').eq('genre_id', g['id']).execute().count or 0
    rows.append({'Genero': g['name'], 'Sets': s, 'Transicoes': t})

df = pd.DataFrame(rows).sort_values('Sets', ascending=False)
print(f'Total: {df["Sets"].sum()} sets | {df["Transicoes"].sum()} transicoes')
display(df)